# Chapter 2 : Working with the Data

## Setting Up macbook GPU

In [4]:
import torch
print(torch.__version__)
print(torch.backends.mps.is_available())


2.7.0
True


In [5]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
# Example tensor operation on MPS
x = torch.rand(5, 3).to(device)
print(x)


tensor([[0.4749, 0.6790, 0.1699],
        [0.8279, 0.2051, 0.7240],
        [0.5521, 0.7854, 0.2855],
        [0.3447, 0.6331, 0.6161],
        [0.1983, 0.5057, 0.7099]], device='mps:0')


## 2.2 Tokenizing text

In [6]:
with open('the-verdict.txt', 'r',encoding='utf-8') as f:
    raw_text = f.read()

In [7]:
len(raw_text)

20479

In [8]:
import re
text = 'Hello, world. This, is a test.'
result = re.split(r'(\s)',text)
print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [9]:
result = re.split(r'([,.]|\s)',text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [10]:
text = 'Hello, world. Is this-- a test?'
result = re.split(r'([,.:;?_!"()\']|--|\s)',text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [12]:
result = re.split(r'([,.:;?_!"()\']|--|\s)',raw_text)
result = [item.strip() for item in result if item.strip()]
preprocessed = result

In [13]:
len(preprocessed)

4690

In [15]:
preprocessed[:10]

['I',
 'HAD',
 'always',
 'thought',
 'Jack',
 'Gisburn',
 'rather',
 'a',
 'cheap',
 'genius']

## 2.3 Converting tokens into token IDs

In [16]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
vocab_size

1130

In [17]:
vocab = {token:integer for integer,token in enumerate(all_words)}
vocab

{'!': 0,
 '"': 1,
 "'": 2,
 '(': 3,
 ')': 4,
 ',': 5,
 '--': 6,
 '.': 7,
 ':': 8,
 ';': 9,
 '?': 10,
 'A': 11,
 'Ah': 12,
 'Among': 13,
 'And': 14,
 'Are': 15,
 'Arrt': 16,
 'As': 17,
 'At': 18,
 'Be': 19,
 'Begin': 20,
 'Burlington': 21,
 'But': 22,
 'By': 23,
 'Carlo': 24,
 'Chicago': 25,
 'Claude': 26,
 'Come': 27,
 'Croft': 28,
 'Destroyed': 29,
 'Devonshire': 30,
 'Don': 31,
 'Dubarry': 32,
 'Emperors': 33,
 'Florence': 34,
 'For': 35,
 'Gallery': 36,
 'Gideon': 37,
 'Gisburn': 38,
 'Gisburns': 39,
 'Grafton': 40,
 'Greek': 41,
 'Grindle': 42,
 'Grindles': 43,
 'HAD': 44,
 'Had': 45,
 'Hang': 46,
 'Has': 47,
 'He': 48,
 'Her': 49,
 'Hermia': 50,
 'His': 51,
 'How': 52,
 'I': 53,
 'If': 54,
 'In': 55,
 'It': 56,
 'Jack': 57,
 'Jove': 58,
 'Just': 59,
 'Lord': 60,
 'Made': 61,
 'Miss': 62,
 'Money': 63,
 'Monte': 64,
 'Moon-dancers': 65,
 'Mr': 66,
 'Mrs': 67,
 'My': 68,
 'Never': 69,
 'No': 70,
 'Now': 71,
 'Nutley': 72,
 'Of': 73,
 'Oh': 74,
 'On': 75,
 'Once': 76,
 'Only': 77,
 '

In [18]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [19]:
tokenizer = SimpleTokenizerv1(vocab)

In [24]:
test_text = """"It's the last he painted, you know,"
            Mrs. Gisburn said with pardonable pride."""

In [25]:
ids = tokenizer.encode(test_text)
ids

[1,
 56,
 2,
 850,
 988,
 602,
 533,
 746,
 5,
 1126,
 596,
 5,
 1,
 67,
 7,
 38,
 851,
 1108,
 754,
 793,
 7]

In [26]:
text = tokenizer.decode(ids)
text

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [28]:
text = tokenizer.decode(tokenizer.encode(test_text))
text

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

## 2.4 Adding special context tokens

In [30]:
text = 'Hello, do you like tea. is this-- a test'
tokenizer.encode(text)

KeyError: 'Hello'

In [31]:
alll_tokens = sorted(list(set(preprocessed)))
alll_tokens.extend(["<|endoftext|>","<|unk|>"])

vocab = {token:integer for integer,token in enumerate(alll_tokens)}

In [32]:
len(vocab)

1132

In [37]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [38]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]

        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text 

In [39]:
tokenizer = SimpleTokenizerV2(vocab)

In [40]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 7, 584, 999, 6, 115, 1131]

In [41]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea. is this -- a <|unk|>'

## 2.5 Byte Pair encoding : GPT,Llama uses BPE

In [44]:
import tiktoken
tiktoken.__version__

'0.9.0'

In [45]:
tokenizer = tiktoken.get_encoding('gpt2')

In [46]:
tokenizer.encode('Hello world')

[15496, 995]

In [47]:
tokenizer.decode(tokenizer.encode("Hello world"))

'Hello world'

In [49]:
text = ("Hello, do you like tea? <|endoftext|> In the sunlit terraces"
        "of someunknownplace.")

tokenizer.encode(text,allowed_special={"<|endoftext|>"})

[15496,
 11,
 466,
 345,
 588,
 8887,
 30,
 220,
 50256,
 554,
 262,
 4252,
 18250,
 8812,
 2114,
 1659,
 617,
 34680,
 5372,
 13]

## 2.6 Data sampling with sliding window

In [51]:
tokenizer

<Encoding 'gpt2'>

In [58]:
with open('the-verdict.txt','r',encoding='utf-8') as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [55]:
enc_text[:10]

[40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138]

In [57]:
enc_sample = enc_text[50:]
len(enc_sample)

5095

In [62]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f'x: {x}')
print(f'y:      {y}')

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [63]:
for i in range(1,context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context,'------->',desired)

[290] -------> 4920
[290, 4920] -------> 2241
[290, 4920, 2241] -------> 287
[290, 4920, 2241, 287] -------> 257


In [67]:
tokenizer.decode([290])

' and'

In [66]:
for i in range(1,context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context),'------->',tokenizer.decode([desired]))

 and ------->  established
 and established ------->  himself
 and established himself ------->  in
 and established himself in ------->  a


In [70]:
import torch
torch.__version__

'2.7.0'

In [89]:

from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


In [95]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [96]:
with open('the-verdict.txt','r', encoding='utf-8') as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)


[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [97]:
next_batch  = next(data_iter)
print(next_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [92]:
len(dataloader)

5141

In [98]:
dataloader = create_dataloader_v1(raw_text, batch_size=8,max_length=4,stride=4,shuffle=False)
data_iter = iter(dataloader)
inputs,targets = next(data_iter)

print(f'Inputs:\n',inputs)
print(f'Targets:\n',targets)


Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## 2.7 creating token embeddings

In [99]:
input_ids = torch.tensor([2,3,5,1])

In [102]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim) #NN layer with random weights

In [103]:
print(embedding_layer)

Embedding(6, 3)


In [104]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [106]:
embedding_layer(torch.tensor([3]))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)

In [107]:
embedding_layer(input_ids)

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)

In [108]:
input_ids

tensor([2, 3, 5, 1])

In [112]:
tokenizer.n_vocab

50257

### Encoding word positions

In [113]:
vocab_size = 50257
output_dim=256 #embddig size

token_embedding_layer = torch.nn.Embedding(vocab_size,output_dim)

In [114]:
token_embedding_layer

Embedding(50257, 256)

In [124]:
max_length = 4 #context length

dataloader = create_dataloader_v1(raw_text,batch_size=8,max_length=max_length,stride=max_length,shuffle=True)

In [125]:
dataloader

In [126]:
data_iter = iter(dataloader)

In [127]:
inputs,targets = next(data_iter)

In [128]:
inputs.shape 

torch.Size([8, 4])

In [130]:
print('Token Ids:\n', inputs)

Token Ids:
 tensor([[  339,  4808,  9776,    62],
        [  339,  2993,   655,   644],
        [ 3363,    11,   340,   373],
        [  198,   198,     1,    40],
        [ 6000,  1517,   373,   284],
        [  290,   673,  1297,  8276],
        [ 1139,  2063,   262,   640],
        [  663,   588,   757, 13984]])


In [131]:
targets.shape

torch.Size([8, 4])

In [133]:
print(targets)

tensor([[ 4808,  9776,    62,  1364],
        [ 2993,   655,   644,   262],
        [   11,   340,   373,   314],
        [  198,     1,    40,  4601],
        [ 1517,   373,   284,   766],
        [  673,  1297,  8276,  2073],
        [ 2063,   262,   640,   407],
        [  588,   757, 13984,   198]])


Converting token ids to embedding vector

In [136]:
token_embeddings = token_embedding_layer(inputs)

In [137]:
token_embeddings.shape

torch.Size([8, 4, 256])

In [139]:
token_embeddings[0].shape

torch.Size([4, 256])

In [141]:
token_embeddings[0,0].shape

torch.Size([256])

In [144]:
token_embeddings[0,0]

tensor([-1.2385, -0.6702, -0.8588,  0.1118,  0.5428, -1.5470,  2.1877, -0.7910,
        -0.2925,  0.5210,  0.8286,  0.0047, -0.1035, -0.5179,  0.2340, -1.5627,
         1.7647,  0.1122,  0.9143,  0.9299, -0.8315,  0.7094, -0.9320, -0.7923,
        -1.3164, -1.6932, -0.1022, -1.8158,  0.2525, -0.0666, -0.1704,  0.3441,
        -1.7072,  0.3912,  0.2580,  0.0997,  0.3414, -0.4195, -0.5311, -0.6472,
        -1.0633,  1.3852, -1.2658, -0.9857,  1.6592, -0.2952,  0.0750, -0.2242,
        -0.6929,  0.2287, -0.9855, -0.9094, -0.8274, -0.4685,  0.2802,  0.7683,
        -1.4124, -0.1138, -1.8953, -0.8651,  0.2830,  1.7703, -0.0339, -0.1696,
         0.7584, -1.0366, -1.5711, -0.8153, -0.8194,  0.8538,  0.4947, -1.4306,
        -0.8505,  0.2685,  1.1033, -0.5106,  1.5691,  0.8500, -0.4160, -0.5450,
        -0.7286, -0.5803,  0.7676, -0.9432, -0.4322,  0.7793, -0.2043, -0.9375,
        -0.1156, -1.0081,  1.2855,  1.0354, -0.8368, -0.3950,  0.9339, -1.6091,
         1.7911,  0.5569, -0.9497, -1.52

In [150]:
contex_length = max_length
pos_embedding_layer = torch.nn.Embedding(contex_length,output_dim)

In [147]:
a = torch.arange(max_length)
a

tensor([0, 1, 2, 3])

In [148]:
pos_embeddings = pos_embedding_layer(a)
print(pos_embeddings.shape)

torch.Size([4, 256])


In [154]:
pos_embedding_layer.weight

Parameter containing:
tensor([[ 1.3903, -0.5993, -0.2518,  ...,  0.0133, -0.2618,  1.0844],
        [-0.6885,  3.1421,  0.6106,  ...,  1.5089, -0.2077,  0.2792],
        [-0.5960, -0.8525, -0.4569,  ...,  0.5257,  0.4323, -0.4627],
        [ 0.1980, -0.4919, -1.1930,  ..., -0.8476,  0.6734,  0.9009]],
       requires_grad=True)

In [156]:
token_embeddings.shape

torch.Size([8, 4, 256])

In [157]:
pos_embeddings.shape

torch.Size([4, 256])

In [158]:
token_embeddings[0] + pos_embeddings

tensor([[ 0.6613, -0.8196, -2.8636,  ...,  1.9964, -1.3287,  1.0141],
        [-2.6648, -1.9005,  0.1842,  ..., -2.7129, -2.9455, -0.5301],
        [-1.5057,  0.2978,  0.1593,  ..., -1.3223,  1.2587, -0.6016],
        [-2.7944, -0.0271,  0.1585,  ...,  2.9305,  2.0949,  1.3478]],
       grad_fn=<AddBackward0>)

In [160]:
input_embeddings = token_embeddings + pos_embeddings

In [161]:
input_embeddings.shape

torch.Size([8, 4, 256])

In [162]:
input_embeddings

tensor([[[ 0.6613, -0.8196, -2.8636,  ...,  1.9964, -1.3287,  1.0141],
         [-2.6648, -1.9005,  0.1842,  ..., -2.7129, -2.9455, -0.5301],
         [-1.5057,  0.2978,  0.1593,  ..., -1.3223,  1.2587, -0.6016],
         [-2.7944, -0.0271,  0.1585,  ...,  2.9305,  2.0949,  1.3478]],

        [[ 0.6613, -0.8196, -2.8636,  ...,  1.9964, -1.3287,  1.0141],
         [-2.1985, -1.4088,  0.4651,  ..., -1.3888, -0.3885,  0.6383],
         [-0.7532,  0.5869,  1.9067,  ...,  1.0685,  3.3874,  0.2053],
         [-2.1543, -1.6636,  0.0053,  ...,  0.8663,  2.5497, -1.8874]],

        [[ 1.6082,  0.3516, -3.1229,  ...,  0.6013, -2.2352,  0.9035],
         [-1.0079, -0.6728,  0.0648,  ..., -1.8819, -2.3177,  0.6696],
         [-0.3375, -1.1894, -0.3766,  ..., -0.7794,  0.5621,  1.4395],
         [-0.6914, -0.1738,  1.0973,  ...,  1.8529,  0.1107,  1.6370]],

        ...,

        [[ 2.2039,  1.1718, -1.1651,  ...,  1.3388, -0.9421, -0.4549],
         [-1.3959, -1.8305,  0.6481,  ..., -1.6483, -1.41